<a href="https://colab.research.google.com/github/Simranjit15kaur/Kth_worldModel/blob/main/Colab3_SpatialTX_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab 3 — Spatial Transformer Dynamics Training
**KTH World Model Pipeline**

**Requires:** `vae_v2.pth` uploaded by Colab 1  
**Can run in parallel with Colab 2** (they train different models)

```
Colab 1 → trains VAE              → uploads vae_v2.pth
Colab 2 → downloads VAE           → trains ConvLSTM    → uploads dynamics_v2.pth
Colab 3 → downloads VAE           → trains Spatial TX  → uploads spatial_transformer.pth
Colab 4 → downloads all 3         → evaluation + GIFs
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CONFIG — same values as Colab 1
# ══════════════════════════════════════════════════════════════════════════
HF_TOKEN   = #entered
HF_REPO_ID = #entered

SPATIAL_EPOCHS = 50    # Spatial TX is slower per-epoch; 50 is sufficient
BATCH_SIZE     = 8
DATA_PATH      = '/content/kth_data'
SAVE_PATH      = '/content'
KTH_ACTIONS    = ['walking', 'jogging']

In [ ]:
!pip install torch torchvision scikit-image imageio tqdm lpips Pillow huggingface_hub -q

In [ ]:
import os, glob, random, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import numpy as np
from PIL import Image
from tqdm import tqdm
import lpips

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

In [ ]:
# ── Download KTH ──────────────────────────────────────────────────────────
KTH_BASE_URL = 'https://www.csc.kth.se/cvap/actions/'

def download_kth_lean(actions, data_path):
    import requests, zipfile
    for action in actions:
        zip_path     = os.path.join(data_path, f'{action}.zip')
        extract_path = os.path.join(data_path, action)
        if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
            print(f'  {action} already exists — skipping'); continue
        print(f'  Downloading {action}...')
        with requests.get(KTH_BASE_URL + f'{action}.zip', stream=True) as r:
            r.raise_for_status()
            with open(zip_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
        with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(data_path)
        os.remove(zip_path)
        print(f'  {action} done.')

download_kth_lean(KTH_ACTIONS, DATA_PATH)

In [ ]:
# ── KTH Dataset (identical to Colab 1) ───────────────────────────────────
class KTHDataset(Dataset):
    IMG_SIZE = 64
    SEQ_LEN  = 20

    def __init__(self, data_path, actions, split='train', augment=True):
        self.augment   = augment
        self.sequences = []
        self.transform = T.Compose([
            T.Grayscale(),
            T.Resize((self.IMG_SIZE, self.IMG_SIZE),
                     interpolation=T.InterpolationMode.LANCZOS, antialias=True),
            T.ToTensor(),
        ])
        train_persons = {f'person{i:02d}' for i in range(1, 17)}
        test_persons  = {f'person{i:02d}' for i in range(17, 26)}
        allowed = train_persons if split == 'train' else test_persons
        all_videos = glob.glob(os.path.join(data_path, '*.avi'))
        print(f'Found {len(all_videos)} total videos')
        for vid_file in sorted(all_videos):
            filename = os.path.basename(vid_file).lower()
            if not any(action in filename for action in actions): continue
            person = filename.split('_')[0]
            if person not in allowed: continue
            frames = self._load_video(vid_file)
            for start in range(0, len(frames) - self.SEQ_LEN, self.SEQ_LEN // 2):
                clip = frames[start:start + self.SEQ_LEN]
                if len(clip) == self.SEQ_LEN: self.sequences.append(clip)
        print(f'KTH {split}: {len(self.sequences)} sequences')

    def _load_video(self, path):
        import cv2
        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            t = self.transform(Image.fromarray(frame))
            t_min, t_max = t.min(), t.max()
            if t_max > t_min: t = (t - t_min) / (t_max - t_min)
            frames.append(t)
        cap.release()
        return frames

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        seq = torch.stack(self.sequences[idx], dim=0)
        if self.augment and random.random() > 0.5:
            seq = torch.flip(seq, dims=[3])
        return seq


train_dataset = KTHDataset(DATA_PATH, KTH_ACTIONS, split='train', augment=True)
test_dataset  = KTHDataset(DATA_PATH, KTH_ACTIONS, split='test',  augment=False)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# ── VAE definition (needed to load and encode) ────────────────────────────
class VAE(nn.Module):
    def __init__(self, latent_channels=128):
        super().__init__()
        self.latent_channels = latent_channels
        self.encoder = nn.Sequential(
            nn.Conv2d(1,   32,  4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(32,  64,  4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(128, latent_channels * 2, 3, stride=1, padding=1),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_channels, 128, 3, stride=1, padding=1), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64,  32, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32,   1, 4, stride=2, padding=1), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        mu, logvar = h.chunk(2, dim=1)
        return mu, torch.clamp(logvar, -10, 10)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def decode(self, z): return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar


# ── Download VAE from Hugging Face ────────────────────────────────────────
from huggingface_hub import hf_hub_download

vae_ckpt_path = hf_hub_download(repo_id=HF_REPO_ID, filename='vae_v2.pth', token=HF_TOKEN)
vae_v2 = VAE().to(device)
ckpt   = torch.load(vae_ckpt_path, map_location=device)
vae_v2.load_state_dict(ckpt['model'])
vae_v2.eval()
for p in vae_v2.parameters(): p.requires_grad = False
print(f'VAE v2 loaded and frozen | Training loss was: {ckpt["losses"][-1]:.4f}')


def encode_sequence(vae, frames_seq):
    B, T = frames_seq.shape[:2]
    return torch.stack([vae.encode(frames_seq[:, t])[0] for t in range(T)], dim=1)

In [ ]:
# ── Spatial Transformer Dynamics ──────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class SpatialTransformerDynamics(nn.Module):
    def __init__(self, latent_channels=128, num_heads=8, num_layers=4, ff_dim=512):
        super().__init__()
        self.pos_enc = PositionalEncoding(latent_channels)
        layer = nn.TransformerEncoderLayer(
            d_model=latent_channels, nhead=num_heads,
            dim_feedforward=ff_dim, batch_first=True, dropout=0.1,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.proj = nn.Linear(latent_channels, latent_channels)

    def forward(self, latent_seq):
        B, T, C, H, W = latent_seq.shape
        tokens = latent_seq.permute(0, 1, 3, 4, 2).reshape(B, T * H * W, C)
        tokens = self.pos_enc(tokens)
        tokens = self.transformer(tokens)
        return self.proj(tokens).reshape(B, T, H, W, C).permute(0, 1, 4, 2, 3)


spatial_model = SpatialTransformerDynamics().to(device)
print(f'Spatial TX | {sum(p.numel() for p in spatial_model.parameters())/1e6:.2f}M params')

In [ ]:
spatial_model     = SpatialTransformerDynamics().to(device)
spatial_optimizer = optim.Adam(spatial_model.parameters(), lr=1e-4)
spatial_scheduler = optim.lr_scheduler.CosineAnnealingLR(spatial_optimizer, T_max=50)

In [ ]:
# ── Reset model + optimizer with better settings ──────────────────────────
spatial_model     = SpatialTransformerDynamics().to(device)
spatial_optimizer = optim.Adam(spatial_model.parameters(), lr=1e-4)
epochs = 55
spatial_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    spatial_optimizer, T_max=epochs)
spatial_losses = []

spatial_model.train()
for epoch in range(epochs):
    epsilon = min(0.5, (epoch / epochs) * 0.5)
    total   = 0

    for batch in tqdm(train_loader, desc=f'SpatialTX Epoch {epoch+1}/{epochs}'):
        batch = batch.to(device)

        with torch.no_grad():
            lat_in  = encode_sequence(vae_v2, batch[:, :10])
            lat_tgt = encode_sequence(vae_v2, batch[:, 10:])

        # ── Single forward pass — no inner loop ───────────────────────────
        # Scheduled sampling: randomly replace some input tokens with
        # previous output. Do it all at once before the forward pass.
        if epsilon > 0:
            with torch.no_grad():
                out_prev = spatial_model(lat_in)   # clean forward for prev preds
            B, T, C, H, W = lat_in.shape
            use_pred = torch.bernoulli(
                torch.ones(B, T, device=device) * epsilon
            ).bool()                               # (B, T) mask
            use_pred[:, 0] = False                 # never replace t=0
            mask = use_pred.view(B, T, 1, 1, 1).expand_as(lat_in)
            lat_in_mixed = torch.where(mask, out_prev.detach(), lat_in)
        else:
            lat_in_mixed = lat_in

        # Single forward pass on the mixed sequence
        pred_latents = spatial_model(lat_in_mixed)

        loss = F.mse_loss(pred_latents, lat_tgt)
        spatial_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(spatial_model.parameters(), 1.0)
        spatial_optimizer.step()
        total += loss.item()

    spatial_scheduler.step()
    avg = total / len(train_loader)
    spatial_losses.append(avg)

    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:02d}/{epochs}  Loss: {avg:.6f}  ε={epsilon:.2f}')
        torch.save({'model': spatial_model.state_dict(),
                    'losses': spatial_losses},
                   f'{SAVE_PATH}/spatial_transformer.pth')
        print('  Checkpoint saved.')

torch.save({'model': spatial_model.state_dict(), 'losses': spatial_losses},
           f'{SAVE_PATH}/spatial_transformer.pth')
print('Spatial Transformer saved.')

In [ ]:
# ── Upload to Hugging Face Hub ────────────────────────────────────────────
from huggingface_hub import HfApi
# path to saved model
SPATIAL_FINAL = f'{SAVE_PATH}/spatial_transformer.pth'
api = HfApi(token='#entered')
api.upload_file(
    path_or_fileobj=SPATIAL_FINAL,
    path_in_repo='spatial_transformer.pth',
    repo_id='SimranjitKaur/vae_kth-world-model',
    repo_type='model',
)
print(f' spatial_transformer.pth uploaded to https://huggingface.co/SimranjitKaur/vae_kth-world-model')

## Colab 3 Complete

Once both Colab 2 and Colab 3 are done, run Colab 4 for evaluation and GIF generation.